# Engine v5 — measure it, and mine negatives with its loose profile

Engine v5 (`js/engine_v5.js`) is v4 plus the fixes from your review notes (`docs/review-notes.md`); the report with all numbers is
`docs/plans/engine-v5-report.md`. This notebook re-runs the measurements and the next step:

| Part | What it does |
|---|---|
| **1 · v3 / v4 / v5** | recall on the 15 reviewed Bin Baz lessons (583 GT quotes) and precision from your labels |
| **2 · Each fix alone** | v5 with one fix switched off at a time: what that fix adds (slow: one engine run per fix) |
| **3 · Precision labels** | Sonnet labels the findings no one labelled yet (store `sonnet_engine_eval`, kept apart from Laya's labels) |
| **4 · Loose profile** | v5 with every bar lowered, on the new lectures: Sonnet labels its findings → many *Not a quote* rows for Laya |

Engine runs are cached per engine file: editing `engine_v5.js` re-runs v5 only, never the speech-to-text.

## Setup

In [ ]:
import sys, os, importlib
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import display
from backend.laya_datagen import engine_eval, autolabel, lectures, stages
for _m in (engine_eval, autolabel, lectures): importlib.reload(_m)   # picks up code edits without restarting the kernel
pd.set_option("display.max_colwidth", 200)

RUN_MODELS = False   # True: Sonnet labels new cards (uses your Claude allowance) · False: saved labels only (safe for Run all)

---
## 1 · v3 / v4 / v5 on the 15 reviewed lessons
**Recall** = GT quotes found (of the alignable ones). **Precision** = real quotes (incl. *by meaning* / *ayah in a hadith*) among the
findings that have a verdict: yours first, then the automatic GT label, then Sonnet. *unlabelled* findings are left out until part 3
labels them. **complete** = GT findings whose span covers the whole quote marked in the written text.

In [ ]:
bundles, ls = engine_eval.load()                      # v3 + v4 records (cached by the review notebook's pipeline)
engine_eval.add_engine(bundles, ls, "v5")
engine_eval.add_engine(bundles, ls, "v5:loose")
ENGINES = ["v3", "v4", "v5", "v5:loose"]
display(pd.DataFrame(engine_eval.report(bundles, ENGINES)))
display(pd.DataFrame(engine_eval.boundaries(bundles, ["v4", "v5"])))

### What v5 changes against v4
GT quotes it now finds / loses, findings it adds, and findings whose span or reference changed.

In [ ]:
d = engine_eval.diff(bundles, "v4", "v5")
print(d["summary"])
print("\nGT quotes v5 finds and v4 missed:")
for it in d["gained"]: print(" ", engine_eval.show_gt(it, bundles))
print("\nNew findings:")
for r in d["added"]: print(" ", engine_eval.show(r, bundles))

---
## 2 · Each fix alone
v5 with one fix switched off: the difference to full v5 is what that fix brings. One engine run per fix (~3 min each).
Names are the switches in `P.fix` at the top of `js/engine_v5.js`.

In [ ]:
FIXES = ["variants", "qul", "free", "muq", "fused", "seed2", "cues", "short", "multi", "ref", "chain", "merge"]
RUN_ABLATION = False          # True: run the 12 engine variants (slow, cached afterwards)
if RUN_ABLATION:
    rows = []
    for f in FIXES:
        engine_eval.add_engine(bundles, ls, f"v5/-{f}", opts={"fix": {f: False}})
        s = engine_eval.diff(bundles, f"v5/-{f}", "v5")["summary"]
        rows.append({"fix": f, **{k: s[k] for k in ("GT gained", "GT lost", "findings added", "added verdicts", "dropped",
                                                     "dropped verdicts", "re-spanned", "other reference")}})
    display(pd.DataFrame(rows))

---
## 3 · Precision labels for spans nobody labelled
Sonnet (same guide and cards as the blind test) labels every finding of v4 / v5 / v5-loose that has no verdict yet. The answers go to
`labels/sonnet_engine_eval.json`, so this evaluation never mixes with Laya's training labels (`labels/sonnet.json`). Re-run part 1 after.

In [ ]:
pool = engine_eval.unlabelled(bundles, ["v4", "v5", "v5:loose"])
print(len(pool), "findings without a verdict")
autolabel.run(pool, bundles, len(pool), RUN_MODELS, store=engine_eval.EVAL_STORE)

---
## 4 · Loose profile on the new lectures → negatives for Laya
`v5:loose` = v5 with `localStorage.wahyEngineProfile = "loose"` (the app's own switch; the app default stays normal). It lowers every
acceptance bar, so it finds many more borderline spans: its precision (part 1) says how loose the app could run with Laya as the
safety net, and its *Not a quote* findings are the training negatives. Lectures: batch 1 and batch 2 of `lectures.py`, only what is
already downloaded and transcribed (`notebooks/laya_lectures.ipynb` does that).

In [ ]:
lec, lec_ls = engine_eval.load_lectures({**lectures.YOUTUBE, **{f"{k} (2)": v for k, v in lectures.YOUTUBE_2.items()}},
                                       lectures.BINBAZ + lectures.BINBAZ_2)
engine_eval.add_engine(lec, lec_ls, "v5:loose")
loose = [r for b in lec.values() for r in b["records"] if r["engine"] == "v5:loose"]
print(f"{len(lec)} lectures · v4 {sum(r['engine'] == 'v4' for b in lec.values() for r in b['records'])} findings · v5 loose {len(loose)} findings")

### Label the loose findings (Laya training labels)
Pool = loose findings with no label yet (none of yours, no Sonnet label, not on a quote marked in Bin Baz's written text). Answers go
to `labels/sonnet.json` with the rest of Laya's labels. `N_CARDS` limits one run; re-running continues.

In [ ]:
pool = [r for r in engine_eval.unlabelled(lec, ["v5:loose"]) if not stages.is_gt_finding(r)]
print(len(pool), "loose findings to label")
N_CARDS = 300
autolabel.run(pool, lec, N_CARDS, RUN_MODELS)

In [ ]:
labelled = engine_eval.label_stores()
v = [engine_eval.verdict(r, labelled)[0] for r in loose]
pd.Series([x or "unlabelled" for x in v]).value_counts()

---
## 5 · All your labels, by reviewer — and how much can train Laya
Every label of an engine finding: yours (stages 2 / 4, negatives sprint), the automatic one from Bin Baz's written text, JEV, Gemini,
base Laya, DeepSeek, Sonnet per effort (model answers are matched to findings through the envelope they were asked about).
**Usable** = a finding with Laya's input, labelled by you / the written text / Sonnet (never JEV, Gemini or base Laya), one label
per finding (you > written text > Sonnet high > Sonnet medium); *places* merges near-duplicate spans of the same quote.

In [ ]:
from backend.laya_datagen import inventory
importlib.reload(inventory)
recs, where = inventory.collect()
ans = inventory.answers(recs)
display(pd.DataFrame(inventory.by_reviewer(ans, recs, where)).fillna(0))
rows = inventory.usable(ans, recs, where)
display(pd.DataFrame([{"set": k, **v} for k, v in inventory.splits(rows).items()]).fillna(""))